# Calculate diff between all the generated json schemas

In [1]:
import os
import json
from jsondiff import diff


In [2]:

def load_all_schemas(root_dir):
    """Finds and loads all detected_schema.json files in the nested tree."""
    schemas = {}
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename == "detected_schema.json":
                # Use the unique folder name as the identifier for this schema
                folder_id = os.path.basename(dirpath)
                filepath = os.path.join(dirpath, filename)
                try:
                    with open(filepath, 'r', encoding='utf-8') as f:
                        schemas[folder_id] = json.load(f)
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
    return schemas

def calculate_structural_distance(schema_a, schema_b):
    """Calculates a rough structural 'distance' based on the number of diff elements."""
    delta = diff(schema_a, schema_b, syntax='explicit')
    
    # Count how many modifications happen at the root level of the diff
    # More keys in the diff = greater evolutionary distance
    distance = len(str(delta))  # Simple string-length proxy for delta complexity
    return distance, delta

def map_schema_topology(root_dir):
    schemas = load_all_schemas(root_dir)
    folder_ids = list(schemas.keys())
    
    print(f"Loaded {len(folder_ids)} schemas. Analyzing relationships...\n")
    
    # Compare every schema to every other schema
    for i, id_a in enumerate(folder_ids):
        closest_neighbor = None
        min_distance = float('inf')
        smallest_delta = None
        
        for j, id_b in enumerate(folder_ids):
            if i == j:
                continue
                
            dist, delta = calculate_structural_distance(schemas[id_a], schemas[id_b])
            if dist < min_distance:
                min_distance = dist
                closest_neighbor = id_b
                smallest_delta = delta
                
        print(f"🌐 Schema [{id_a}] is closest to [{closest_neighbor}]")
        # Optional: Print out the tiny delta to see the single evolutionary step
        # print(json.dumps(smallest_delta, indent=2))

if __name__ == "__main__":
    # Point this to your clustered/nested root directory
    map_schema_topology("clustered_ai-json-files")

Loaded 212 schemas. Analyzing relationships...

🌐 Schema [schema_group_59_23d67771] is closest to [schema_group_96_da5cc9f2]
🌐 Schema [schema_group_115_209e3ed5] is closest to [schema_group_34_b5c93539]
🌐 Schema [schema_group_193_5b36d10e] is closest to [schema_group_57_2c547c0d]
🌐 Schema [schema_group_41_a7be8828] is closest to [schema_group_68_d434a8c2]
🌐 Schema [schema_group_7_c9b8a033] is closest to [schema_group_34_b5c93539]
🌐 Schema [schema_group_105_a71434ec] is closest to [schema_group_130_ef25e4c3]
🌐 Schema [schema_group_63_cdfd3a3c] is closest to [schema_group_41_a7be8828]
🌐 Schema [schema_group_99_d6f60ad0] is closest to [schema_group_96_da5cc9f2]
🌐 Schema [schema_group_68_d434a8c2] is closest to [schema_group_34_b5c93539]
🌐 Schema [schema_group_25_e21682d1] is closest to [schema_group_18_6934c671]
🌐 Schema [schema_group_15_87f9c798] is closest to [schema_group_29_4f7073a2]
🌐 Schema [schema_group_188_818409d0] is closest to [schema_group_57_2c547c0d]
🌐 Schema [schema_group_2

## Consolidate all detected_schema.json into one directory.

In [4]:
import os
import shutil

def consolidate_and_rename_schemas(root_dir, target_dir="json_schema"):
    """
    Traverses a nested directory structure, finds 'detected_schema.json' files,
    and copies them to a flat target directory renamed as 'detected_<parent_folder>.json'.
    
    :param root_dir: The root directory containing your nested folders (e.g., 'xyz_clustered')
    :param target_dir: The destination directory where all renamed schemas will be stored
    """
    # Create the target directory if it doesn't exist yet
    os.makedirs(target_dir, exist_ok=True)
    
    copied_count = 0
    print(f"Starting schema consolidation from '{root_dir}' into '{target_dir}'...\n")

    # Walk through the nested directory structure
    for dirpath, _, filenames in os.walk(root_dir):
        for filename in filenames:
            # Look specifically for your target filename
            if filename == "detected_schema.json":
                source_filepath = os.path.join(dirpath, filename)
                
                # Extract the immediate parent directory name (<src_folder>)
                src_folder_name = os.path.basename(dirpath)
                
                # Construct the new filename: detected_<src_folder>.json
                new_filename = f"detected_{src_folder_name}.json"
                target_filepath = os.path.join(target_dir, new_filename)
                
                try:
                    # Copy and rename the file
                    shutil.copy2(source_filepath, target_filepath)
                    print(f"  -> Copied: {source_filepath}")
                    print(f"     Saved as: {target_filepath}\n")
                    copied_count += 1
                except IOError as e:
                    print(f"  ![Error] Failed to copy {source_filepath}: {e}")

    print(f"--- Process Complete ---")
    print(f"Successfully consolidated {copied_count} schemas into '{target_dir}'.")

if __name__ == "__main__":
    # Replace 'xyz_clustered' with the actual path to your nested directory
    SOURCE_ROOT = "sat_clustered"  # Change this to your actual source root directory
    TARGET_DIRECTORY = "consolidated_sat_schemas"  # Change this to your desired target directory
    
    consolidate_and_rename_schemas(SOURCE_ROOT, TARGET_DIRECTORY)

Starting schema consolidation from 'sat_clustered' into 'consolidated_sat_schemas'...

  -> Copied: sat_clustered/schema_group_2_40d68f79/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_2_40d68f79.json

  -> Copied: sat_clustered/schema_group_4_adbd08df/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_4_adbd08df.json

  -> Copied: sat_clustered/schema_group_6_cf7b7d95/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_6_cf7b7d95.json

  -> Copied: sat_clustered/schema_group_5_983cf819/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_5_983cf819.json

  -> Copied: sat_clustered/schema_group_8_99da8e24/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_8_99da8e24.json

  -> Copied: sat_clustered/schema_group_1_e75e7511/detected_schema.json
     Saved as: consolidated_sat_schemas/detected_schema_group_1_e75e7511.json

  -> Co